<a href="https://colab.research.google.com/github/Jennymmmmm/lis5693/blob/main/lab-8/Lab-8_Jenny%20Min.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%capture
!pip install bertopic datasets openai datamapplot

In [ ]:
!pip install bertopic

In [ ]:
from datasets import load_dataset

dataset = load_dataset("csv", data_files="/content/lens-export.csv")["train"]

# Filter out None values from abstracts and keep titles in sync
filtered_abstracts_and_titles = [
    (abstract, title)
    for abstract, title in zip(dataset["Abstract"], dataset["Title"])
    if abstract is not None
]

abstracts = [item[0] for item in filtered_abstracts_and_titles]
titles = [item[1] for item in filtered_abstracts_and_titles]

In [ ]:
from sentence_transformers import SentenceTransformer

# Create an embedding for each abstract
embedding_model = SentenceTransformer('thenlper/gte-small')
embeddings = embedding_model.encode(abstracts, show_progress_bar=True)

In [ ]:
# Check the dimensions of the resulting embeddings
embeddings.shape

In [ ]:
from umap import UMAP

# We reduce the input embeddings from 605 dimenions to 5 dimenions
umap_model = UMAP(
    n_components=5, min_dist=0.0, metric='cosine', random_state=42
)
reduced_embeddings = umap_model.fit_transform(embeddings)

In [ ]:
from hdbscan import HDBSCAN

# We fit the model and extract the clusters
hdbscan_model = HDBSCAN(
    min_cluster_size=50, metric='euclidean', cluster_selection_method='eom'
).fit(reduced_embeddings)
clusters = hdbscan_model.labels_

# How many clusters did we generate?
len(set(clusters))

In [ ]:
import numpy as np

# Print first three documents in cluster 0
cluster = 0
for index in np.where(clusters==cluster)[0][:3]:
    print(abstracts[index][:300] + "... \n")

In [ ]:
import pandas as pd

# Reduce 384-dimensional embeddings to 2 dimensions for easier visualization
reduced_embeddings = UMAP(
    n_components=2, min_dist=0.0, metric='cosine', random_state=42
).fit_transform(embeddings)

# Create dataframe
df = pd.DataFrame(reduced_embeddings, columns=["x", "y"])
df["title"] = titles
df["cluster"] = [str(c) for c in clusters]

# Select outliers and non-outliers (clusters)
clusters_df = df.loc[df.cluster != "-1", :]
outliers_df = df.loc[df.cluster == "-1", :]

In [ ]:
import matplotlib.pyplot as plt

# Plot outliers and non-outliers seperately
plt.scatter(outliers_df.x, outliers_df.y, alpha=0.05, s=2, c="grey")
plt.scatter(
    clusters_df.x, clusters_df.y, c=clusters_df.cluster.astype(int),
    alpha=0.6, s=2, cmap='tab20b'
)
plt.axis('off')
# plt.savefig("matplotlib.png", dpi=300)  # Uncomment to save the graph as a .png

In [ ]:
from hdbscan import HDBSCAN
from umap import UMAP

umap_model = UMAP(
    n_components=5, min_dist=0.0, metric='cosine', random_state=42
)

hdbscan_model = HDBSCAN(
    min_cluster_size=10,  # lower = more topics
    min_samples=3,
    prediction_data=True
)

vectorizer_model = CountVectorizer(stop_words="english")

topic_model = BERTopic(
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
)

topics, probs = topic_model.fit_transform(abstracts, embeddings)

By adjusting min_cluster_size in HDBSCAN (shown above), the model was able to detect more granular topics. The previous configuration only produced 3 topics (shown below), whereas the updated parameters resulted in more distinct and meaningful clusters, leading to clearer visualizations!


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from bertopic import BERTopic

vectorizer_model = CountVectorizer(stop_words="english")

topic_model = BERTopic(
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    nr_topics=10
)

topics, probs = topic_model.fit_transform(abstracts, embeddings)

In [ ]:
topic_model.get_topic_info()

In [ ]:
topic_model.get_topic(0)

In [ ]:
topic_model.get_topic(0)
topic_model.get_topic(1)

In [ ]:
topic_model.topics_[titles.index('Early childhood education and child development outcomes in least developed countries: empirical evidence from Lao PDR')]

In [ ]:
# Visualize topics and documents
fig = topic_model.visualize_documents(
    titles,
    reduced_embeddings=reduced_embeddings,
    width=1200,
    hide_annotations=True
)

# Update fonts of legend for easier visualization
fig.update_layout(font=dict(size=16))

In [ ]:
# Visualize barchart with ranked keywords
topic_model.visualize_barchart()

# Visualize relationships between topics
topic_model.visualize_heatmap(n_clusters=2)

# Visualize the potential hierarchical structure of topics
topic_model.visualize_hierarchy()

**Q:  Interpret and compare your results with those obtained in  lab-5. What similarities and differences can be observed between topic modeling using BERTopic and LDA, and what underlying factors might explain these outcomes?**

Compared to the LDA results from Lab 5, BERTopic did a much better job at breaking down the broad topic of early childhood education into more specific subtopics. While LDA only produced a few topics that mostly shared the same general terms, BERTopic was able to identify 20 distinct topics ranging from physical education and STEM to sustainability, inclusive education, and even region-specific research from countries like China, Australia, India, and Korea.
Both methods did pick up on some of the same themes, such as teacher roles and child development came through in both, which makes sense given the nature of the dataset. The difference of those two is that BERTopic went much deeper than LDA, separating out subtopics that LDA lumped together or missed entirely. I assume this to be likely because BERTopic understands the meaning behind words rather than just counting how often they appear together, which gives it an edge when dealing with a dataset that's focused on one broad subject area.

The main challenge I encountered today was getting a sufficient number of topics. Initially, the model only produced 3 topics, which was not very informative given the size of the dataset. I had to experiment with the HDBSCAN parameters, particularly lowering the min_cluster_size, to get the model to detect more granular clusters. It was also a bit time-consuming, as running UMAP and fitting the model on nearly 1,000 documents took more than 15 minutes each time I made changes and had to rerun the pipeline.

As a novice researcher who has a big interest in big data, I think LLM will give me a big opportunity in terms of applying topic modeling to large collections of academic papers and enable me to quickly map out the landscape of the field.  For example, embeddings could be used to track how research priorities in early childhood education have shifted over time, or to compare research trends across different countries and regions. Moreover, this kind of big data analysis could ultimately help policymakers and educators make more informed, evidence-based decisions about curriculum, teacher training, and early intervention programs.